# Chapter 12: Alignment

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch12_alignment.ipynb)


## What is in this notebook, and what to change in it

Three cells, three loss functions, and no model anywhere. Every input is a
tensor of numbers written into the cell, which is deliberate: these are the
objectives, separated from the machinery that optimises them. Nothing here is
random, so every number is the same on every machine.

1. **The SFT loss with the prompt masked out**, so only response tokens
   contribute to the gradient. The cell defines the function and does not call
   it, because calling it needs a real model and a tokenized batch, and the
   usage sits in a comment at the bottom. Reading the mask arithmetic is the
   exercise.
2. **The Bradley-Terry reward-model loss** over eight preference pairs, with
   pairwise accuracy printed beside it. Three of the eight pairs are scored the
   wrong way round by the toy rewards, so the accuracy prints as 62.5 percent
   rather than 100, which is what makes the loss value worth looking at.
3. **The DPO loss** over four preference pairs, with policy and reference log
   probabilities supplied directly.

The edit worth making is in cell 3. Raise beta from 0.1 to 1.0 and rerun: beta
controls how far the policy may move from the reference, and the loss responds
to it sharply. Then set the policy log probabilities equal to the reference
ones. The loss goes to log 2, about 0.693, which is what this objective returns
when the policy has learned nothing at all. Seeing that number appear is worth
more than reading the derivation.


> **This notebook was executed when it was built**, so the output under each
> cell is a real run's and you can read the file without running anything.
> Rebuild it with `python tools/build_notebook.py ch12`.


### 12.2.1 Supervised Fine-Tuning on Instructions


In [1]:
import torch
from torch.nn import CrossEntropyLoss

def sft_loss(model, input_ids, labels, prompt_len):
    """Compute SFT loss on response tokens only (mask the prompt)."""
    logits = model(input_ids).logits          # (batch, seq, vocab)
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    # Only compute loss on response tokens, not prompt tokens
    loss_mask = torch.zeros_like(shift_labels, dtype=torch.float)
    loss_mask[:, prompt_len - 1:] = 1.0
    loss_fn = CrossEntropyLoss(reduction='none')
    token_losses = loss_fn(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )
    token_losses = token_losses.view(shift_labels.shape)
    return (token_losses * loss_mask).sum() / loss_mask.sum()

# Usage (with a real model and tokenized batch):
#   loss = sft_loss(model, input_ids, labels, prompt_len=32)
#   print(f"SFT loss: {loss.item():.4f}")
# Only response tokens contribute to the gradient.


### 12.3.2 Training the Reward Model (Bradley-Terry)


In [2]:
import torch
import torch.nn.functional as F

def reward_model_loss(rewards_preferred, rewards_dispreferred):
    """Bradley-Terry loss for reward model training.
    L_RM = -E[log sigma(r(y_w) - r(y_l))]"""
    return -F.logsigmoid(rewards_preferred - rewards_dispreferred).mean()

# Toy example: 8 preference pairs
r_win  = torch.tensor([2.1, 1.5, 3.0, 0.8, 2.5, 1.9, 2.7, 1.2])
r_lose = torch.tensor([1.0, 1.8, 0.5, 0.2, 1.1, 2.5, 0.3, 1.5])
loss = reward_model_loss(r_win, r_lose)
accuracy = (r_win > r_lose).float().mean()
print(f"RM loss: {loss:.4f}, Pairwise accuracy: {accuracy:.1%}")
# Loss is low when preferred responses have higher reward


RM loss: 0.4821, Pairwise accuracy: 62.5%


### 12.4.2 The DPO Loss Derivation


In [3]:
import torch
import torch.nn.functional as F

def dpo_loss(pi_logprobs_w, pi_logprobs_l,
             ref_logprobs_w, ref_logprobs_l, beta=0.1):
    """Direct Preference Optimization loss (Rafailov et al., 2023).
    L_DPO = -E[log sigma(beta * (log(pi/ref)_w - log(pi/ref)_l))]"""
    log_ratio_w = pi_logprobs_w - ref_logprobs_w   # log(pi(y_w)/pi_ref(y_w))
    log_ratio_l = pi_logprobs_l - ref_logprobs_l   # log(pi(y_l)/pi_ref(y_l))
    logits = beta * (log_ratio_w - log_ratio_l)
    return -F.logsigmoid(logits).mean()

# Toy: sequence log-probs of preferred/dispreferred under policy and reference
pi_w  = torch.tensor([-2.0, -1.5, -1.8, -2.2])   # policy on preferred
pi_l  = torch.tensor([-3.0, -2.8, -3.5, -2.0])   # policy on dispreferred
ref_w = torch.tensor([-2.5, -2.0, -2.3, -2.5])   # reference on preferred
ref_l = torch.tensor([-2.8, -2.5, -3.0, -2.2])   # reference on dispreferred

loss = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
print(f"DPO loss: {loss:.4f}")
# Low loss: policy increases probability of preferred over dispreferred
# relative to the reference model


DPO loss: 0.6613


---

## Summary

This notebook demonstrated the key code examples from Chapter 12: Alignment. For the full mathematical exposition and discussion, refer to the textbook chapter.
